# Generación de Embeddings BGE-M3 → Pinecone (v2: robustez + backup)

Genera embeddings densos de **1024 dimensiones** con `BAAI/bge-m3` y los sube
por lotes a un índice de **Pinecone**.

---
### Qué cambia respecto a la v1

1. **Reintentos con backoff exponencial en el `upsert`** (`upsert_with_backoff`):
   los fallos de red o rate-limiting ya no matan el proceso a mitad de camino.
2. **Manejo de OOM en la GPU** (`embed_batch_with_oom_retry`): si un lote no
   cabe en memoria, se reduce el `batch_size` a la mitad automáticamente y se
   reintenta, con `torch.cuda.empty_cache()` + `gc.collect()` entre intentos.
   Se libera cache de GPU también después de cada lote normal.
3. **Backup local incremental y reanudable** (`noticias_embeddings_backup.jsonl`):
   cada lote de embeddings se escribe en disco **en cuanto se genera**, antes
   de intentar subirlo a Pinecone. Si el notebook se interrumpe y se vuelve a
   ejecutar, los nodos ya presentes en el backup **no se recalculan** — se
   reutilizan directamente, así que nunca se repite cómputo de GPU ya hecho.
   El `upsert` a Pinecone, en cambio, siempre se reintenta para todos los
   nodos en cada ejecución (es una operación idempotente y barata: subir de
   nuevo un vector con el mismo `id` simplemente lo sobrescribe).
4. **Limpieza de metadatos endurecida** (`build_metadata`): además de
   convertir `None` a `""`/`0.0` (ya existía), ahora también filtra `NaN`/
   `Infinity` en campos numéricos, convierte estructuras anidadas
   (dict/list) a JSON-string en vez de dejarlas pasar sin más (Pinecone las
   rechazaría), y aplica un recorte de seguridad al campo `text` por si
   algún nodo superase el límite de metadata por vector de Pinecone (40 KB).
5. **Celdas nuevas al final**: restauración del índice directamente desde el
   backup local (sin GPU, sin volver a descargar el JSON de nodos) y
   consolidación del backup a un único `.parquet` compacto para almacenamiento
   a largo plazo.

## 1. Instalación de dependencias

In [ ]:
# FlagEmbedding: wrapper oficial de BGE-M3 con soporte de embeddings densos/sparse/colbert
# pinecone: cliente oficial v3+
# pyarrow: para consolidar el backup a formato Parquet (celda final, opcional)
!pip install -q FlagEmbedding pinecone pyarrow
print('✅ Dependencias instaladas')

## 2. Importaciones

In [ ]:
import gc
import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
from FlagEmbedding import BGEM3FlagModel
from pinecone import Pinecone, ServerlessSpec
from google.colab import files, userdata
from tqdm.notebook import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM total: {props.total_memory / 1024**3:.1f} GB')

## 3. Configuración

Ajusta los parámetros de esta celda antes de ejecutar el notebook.

> **API Key**: guarda `PINECONE_API_KEY` como secret de Colab (icono 🔑 en la barra lateral).

**Sobre `BATCH_EMBED=32` en una T4 (16 GB VRAM):** con BGE-M3 en fp16
(~2.3 GB de pesos) y `MAX_LENGTH=512`, un batch de 32 se mantiene muy por
debajo del límite de memoria de una T4 en el caso normal — no hace falta
bajarlo de partida. Aun así, `embed_batch_with_oom_retry` (celda 7) actúa
como red de seguridad: si algún lote puntual (p. ej. textos inusualmente
largos) provoca un `OutOfMemoryError`, el batch se parte a la mitad
automáticamente en vez de abortar el notebook.

In [ ]:
# ── Modelo ───────────────────────────────────────────────────────────────────
MODEL_NAME   = 'BAAI/bge-m3'
EMBED_DIM    = 1024          # dimensión de los embeddings densos de BGE-M3
BATCH_EMBED  = 32            # nodos por lote en la inferencia (red de seguridad OOM: se auto-reduce si hace falta)
MIN_BATCH_EMBED = 1          # batch minimo antes de rendirse con un OOM irrecuperable
MAX_LENGTH   = 512           # tokens máximos por chunk (coincide con chunk_size del splitter)

# ── Pinecone ─────────────────────────────────────────────────────────────────
PINECONE_API_KEY  = userdata.get('PINECONE_API_KEY')   # secret de Colab
INDEX_NAME        = 'noticias-financieras'
PINECONE_CLOUD    = 'aws'       # 'aws' | 'gcp' | 'azure'
PINECONE_REGION   = 'us-east-1'
BATCH_UPSERT      = 100         # vectores por lote en el upsert

# ── Reintentos / backoff ────────────────────────────────────────────────────
MAX_RETRIES      = 5     # intentos maximos por lote antes de abortar
RETRY_BASE_DELAY = 1.0   # segundos, se duplica en cada reintento (1, 2, 4, 8, 16...)
RETRY_MAX_DELAY  = 60.0  # techo de espera entre reintentos

# ── Backup local ─────────────────────────────────────────────────────────────
BACKUP_FILE = 'noticias_embeddings_backup.jsonl'

# ── Filtro de nodos a indexar ────────────────────────────────────────────────
# Se indexan TODOS los nodos. Los nodos con enrich_status != 'ok' carecen de
# precios y retornos, pero conservan 'regimen_mercado' (bearish/bullish/sideways)
# que es el metadato de clasificación principal.
FILTER_ENRICH_STATUS = None   # None = sin filtro

print('Configuración cargada ✅')
print(f'  Modelo        : {MODEL_NAME}')
print(f'  Dimensión     : {EMBED_DIM}')
print(f'  Índice        : {INDEX_NAME}')
print(f'  Cloud/Región  : {PINECONE_CLOUD} / {PINECONE_REGION}')
print(f'  Backup local  : {BACKUP_FILE}')

## 4. Carga del archivo de nodos

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, 'r', encoding='utf-8') as f:
    all_nodes: List[Dict[str, Any]] = json.load(f)

print(f'Nodos totales cargados : {len(all_nodes)}')
print(f'Campos disponibles     : {list(all_nodes[0].keys())}')

## 5. Preparación de nodos

Se indexan **todos los nodos** sin excepción, salvo que carezcan de texto
(defensivo: un nodo sin `text` o con `text` vacío no puede generar un
embedding útil y rompería `model.encode`).

In [ ]:
from collections import Counter

nodes = [n for n in all_nodes if (n.get('text') or '').strip()]
skipped = len(all_nodes) - len(nodes)
if skipped:
    print(f'[aviso] {skipped} nodos sin texto util fueron omitidos.')

if FILTER_ENRICH_STATUS is not None:
    nodes = [n for n in nodes if n.get('enrich_status') == FILTER_ENRICH_STATUS]

print(f'Nodos a indexar: {len(nodes)}')
print(Counter(n.get('enrich_status') for n in nodes))

## 6. Carga del modelo BGE-M3

`BGEM3FlagModel` descarga `BAAI/bge-m3` (~2.3 GB) desde Hugging Face la
primera vez que se ejecuta.

In [ ]:
print(f'Cargando {MODEL_NAME} en {DEVICE}...')
t0 = time.time()

model = BGEM3FlagModel(
    MODEL_NAME,
    use_fp16=True,          # fp16 en GPU → mitad de VRAM, misma calidad práctica
    device=DEVICE,
)

print(f'Modelo cargado en {time.time()-t0:.1f}s ✅')

## 7. Funciones de utilidad

Modularizadas para que el bucle principal (celda 8) quede legible:

- `embed_batch_with_oom_retry`: genera embeddings con reintento adaptativo
  ante `OutOfMemoryError` (parte el lote a la mitad recursivamente).
- `build_metadata`: limpia un nodo a un dict compatible con Pinecone
  (sin `None`, sin `NaN`/`Infinity`, sin estructuras anidadas).
- `append_backup` / `load_backup`: I/O del backup local en formato
  [JSON Lines](https://jsonlines.org/) — cada línea es un vector completo
  (`id` + `values` + `metadata`), lo que permite escribir de forma
  incremental (una línea por vector, sin reescribir el archivo entero) y
  reanudar sin ambigüedad si el notebook se interrumpe a mitad de un lote.
- `upsert_with_backoff`: sube un lote a Pinecone con reintentos y espera
  exponencial + jitter ante errores de red/rate-limiting.

In [ ]:
import numpy as np


def embed_batch_with_oom_retry(model, texts, batch_size, max_length, min_batch_size=MIN_BATCH_EMBED):
    '''Genera embeddings densos para "texts" con manejo adaptativo de OOM: si
    CUDA se queda sin memoria, reduce el batch a la mitad y reintenta,
    liberando cache de GPU entre intentos. Devuelve un numpy array (N, EMBED_DIM).'''
    try:
        output = model.encode(
            texts,
            batch_size=batch_size,
            max_length=max_length,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        return output['dense_vecs']
    except RuntimeError as e:
        if 'out of memory' not in str(e).lower():
            raise
        torch.cuda.empty_cache()
        gc.collect()
        if batch_size <= min_batch_size or len(texts) <= min_batch_size:
            raise RuntimeError(
                f'OOM incluso con batch_size={batch_size}. Reduce MAX_LENGTH o '
                'usa una GPU con mas VRAM.'
            ) from e
        new_batch_size = max(min_batch_size, batch_size // 2)
        print(f'  [OOM] batch_size {batch_size} -> {new_batch_size}, reintentando...')
        mid = len(texts) // 2
        first  = embed_batch_with_oom_retry(model, texts[:mid], new_batch_size, max_length, min_batch_size)
        second = embed_batch_with_oom_retry(model, texts[mid:], new_batch_size, max_length, min_batch_size)
        return np.concatenate([first, second], axis=0)


MAX_METADATA_TEXT_BYTES = 30_000  # margen de seguridad bajo el limite de 40KB/vector de Pinecone


def _clean_str(v, default=''):
    if v is None:
        return default
    if isinstance(v, (dict, list)):
        # Pinecone no admite estructuras anidadas en metadata (solo listas de str)
        return json.dumps(v, ensure_ascii=False)
    return str(v)


def _clean_float(v, default=0.0):
    if v is None:
        return default
    try:
        f = float(v)
    except (TypeError, ValueError):
        return default
    if math.isnan(f) or math.isinf(f):
        return default
    return f


def _clean_text(v, default=''):
    s = _clean_str(v, default)
    if len(s.encode('utf-8')) > MAX_METADATA_TEXT_BYTES:
        # recorte defensivo; no deberia activarse con chunk_size=512, es una salvaguarda
        s = s.encode('utf-8')[:MAX_METADATA_TEXT_BYTES].decode('utf-8', errors='ignore')
    return s


def build_metadata(node: Dict[str, Any]) -> Dict[str, Any]:
    '''Construye el dict de metadatos para Pinecone.

    Pinecone solo admite str, int, float, bool y listas de str: ni None, ni
    NaN/Infinity, ni dict/list anidados sobreviven a esta funcion sin
    convertirse a un tipo permitido.
    '''
    return {
        # ── Navegación entre nodos ──────────────────────────────────────────
        'source_doc_id'     : _clean_str(node.get('source_doc_id')),
        'prev_node_id'      : _clean_str(node.get('prev_node_id')),
        'next_node_id'      : _clean_str(node.get('next_node_id')),
        'chunk_size'        : int(node.get('chunk_size', 512) or 512),
        'chunk_overlap'     : int(node.get('chunk_overlap', 64) or 64),
        # ── Metadatos editoriales ───────────────────────────────────────────
        'titulo'            : _clean_str(node.get('titulo')),
        'url'               : _clean_str(node.get('url')),
        'fecha'             : _clean_str(node.get('fecha')),
        'periodo'           : _clean_str(node.get('periodo')),
        'language'          : _clean_str(node.get('language')),
        'indice_sector'     : _clean_str(node.get('indice_sector')),
        'regimen_mercado'   : _clean_str(node.get('regimen_mercado')),
        # ── Metadatos financieros ───────────────────────────────────────────
        'ticker'            : _clean_str(node.get('ticker')),
        'price_t0'          : _clean_float(node.get('price_t0')),
        'price_t1'          : _clean_float(node.get('price_t1')),
        'price_t5'          : _clean_float(node.get('price_t5')),
        'price_t20'         : _clean_float(node.get('price_t20')),
        'return_1d'         : _clean_float(node.get('return_1d')),
        'return_5d'         : _clean_float(node.get('return_5d')),
        'return_20d'        : _clean_float(node.get('return_20d')),
        'market_reaction_1d' : _clean_str(node.get('market_reaction_1d')),
        'market_reaction_5d' : _clean_str(node.get('market_reaction_5d')),
        'market_reaction_20d': _clean_str(node.get('market_reaction_20d')),
        'enrich_status'     : _clean_str(node.get('enrich_status')),
        'enrich_error'      : _clean_str(node.get('enrich_error')),
        # ── Texto del chunk (para recuperación sin ir al JSON) ──────────────
        'text'              : _clean_text(node.get('text')),
    }


def append_backup(records: List[Dict[str, Any]], backup_file: str) -> None:
    '''Anade una lista de registros {id, values, metadata} al backup local en
    formato JSON Lines. Append seguro: cada llamada abre en modo "a", asi que
    es resistente a interrupciones (no reescribe lo ya guardado).'''
    with open(backup_file, 'a', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def load_backup(backup_file: str) -> Dict[str, Dict[str, Any]]:
    '''Carga el backup existente (si lo hay) a un dict {id: {id, values, metadata}}.
    Permite reanudar sin recalcular embeddings ya generados en una ejecucion previa.'''
    backed_up: Dict[str, Dict[str, Any]] = {}
    p = Path(backup_file)
    if not p.exists():
        return backed_up
    with open(p, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                backed_up[rec['id']] = rec
            except (json.JSONDecodeError, KeyError):
                print(f'  [aviso] linea {line_num} del backup corrupta, se ignora')
    return backed_up


def upsert_with_backoff(index, vectors, max_retries=MAX_RETRIES,
                         base_delay=RETRY_BASE_DELAY, max_delay=RETRY_MAX_DELAY):
    '''Sube un lote de vectores a Pinecone con reintentos y backoff exponencial
    + jitter ante errores de red o rate-limiting. Lanza la excepcion original
    si se agotan los reintentos.'''
    attempt = 0
    while True:
        try:
            return index.upsert(vectors=vectors)
        except Exception as e:
            attempt += 1
            if attempt > max_retries:
                raise RuntimeError(
                    f'Fallo el upsert tras {max_retries} reintentos: {type(e).__name__}: {e}'
                ) from e
            delay = min(max_delay, base_delay * (2 ** (attempt - 1)))
            delay += random.uniform(0, delay * 0.1)  # jitter
            print(f'  [reintento {attempt}/{max_retries}] {type(e).__name__}: {e}. '
                  f'Esperando {delay:.1f}s...')
            time.sleep(delay)


print('Funciones de utilidad definidas ✅')

## 8. Generación de embeddings + backup incremental

Por cada lote: **(1)** genera los embeddings, **(2)** los escribe
inmediatamente en `BACKUP_FILE`, **(3)** libera cache de GPU. El backup se
escribe *antes* de intentar nada con Pinecone — si el índice o la conexión
fallan más adelante, los embeddings ya están a salvo en disco.

**Reanudación:** si `BACKUP_FILE` ya contiene vectores de una ejecución
anterior (p. ej. el notebook se cortó a mitad de proceso), esos nodos se
detectan y se **omiten** de la regeneración — se reutilizan directamente
desde el backup. Solo se gasta GPU en los nodos que de verdad faltan.

In [ ]:
backed_up = load_backup(BACKUP_FILE)
print(f'Vectores ya respaldados localmente: {len(backed_up)}' if backed_up
      else 'No hay backup previo, se parte de cero.')

pending_nodes = [n for n in nodes if n['node_id'] not in backed_up]
print(f'Nodos que requieren generar embedding: {len(pending_nodes)} de {len(nodes)}')

if pending_nodes:
    n_batches = math.ceil(len(pending_nodes) / BATCH_EMBED)
    t0 = time.time()

    for i in tqdm(range(n_batches), desc='Embedding + backup', unit='batch'):
        batch_nodes = pending_nodes[i * BATCH_EMBED : (i + 1) * BATCH_EMBED]
        batch_texts = [n['text'] for n in batch_nodes]

        dense_vecs = embed_batch_with_oom_retry(model, batch_texts, BATCH_EMBED, MAX_LENGTH)

        records = [
            {
                'id'      : n['node_id'],
                'values'  : vec.tolist() if hasattr(vec, 'tolist') else list(vec),
                'metadata': build_metadata(n),
            }
            for n, vec in zip(batch_nodes, dense_vecs)
        ]
        append_backup(records, BACKUP_FILE)
        for r in records:
            backed_up[r['id']] = r

        del dense_vecs
        torch.cuda.empty_cache()  # libera fragmentacion acumulada entre lotes

    elapsed = time.time() - t0
    print(f'\n✅ {len(pending_nodes)} embeddings nuevos generados y respaldados en {elapsed:.1f}s '
          f'({elapsed/len(pending_nodes)*1000:.1f} ms/nodo)')
else:
    print('Todos los nodos ya estaban en el backup, no hace falta generar nada nuevo.')

print(f'Total de vectores disponibles (nuevos + backup previo): {len(backed_up)}')
print(f'Dimensión comprobada: {len(next(iter(backed_up.values()))["values"])}')

## 9. Creación del índice Pinecone

Se crea el índice si no existe. Si ya existe con la misma configuración, se reutiliza.

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing:
    print(f'Creando índice "{INDEX_NAME}"...')
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric='cosine',
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION,
        ),
    )
    while not pc.describe_index(INDEX_NAME).status['ready']:
        print('  Esperando...', end='\r')
        time.sleep(3)
    print(f'Índice "{INDEX_NAME}" creado ✅')
else:
    print(f'Índice "{INDEX_NAME}" ya existe, se reutiliza ✅')

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

## 10. Upsert de vectores a Pinecone

Sube **todos** los nodos disponibles en `backed_up` (recién generados +
recuperados de una ejecución anterior), con reintentos y backoff exponencial
ante fallos de red o rate-limiting. El upsert es idempotente: repetir el
mismo `id` simplemente sobrescribe el vector, así que no hay riesgo en
volver a subir nodos que ya estuvieran en el índice.

In [ ]:
all_ids = [n['node_id'] for n in nodes]
n_batches_upsert = math.ceil(len(all_ids) / BATCH_UPSERT)
total_upserted = 0
missing_total = 0
t0 = time.time()

print(f'Subiendo {len(all_ids)} vectores a Pinecone · {n_batches_upsert} lotes · batch={BATCH_UPSERT}')

for i in tqdm(range(n_batches_upsert), desc='Upsert', unit='batch'):
    batch_ids = all_ids[i * BATCH_UPSERT : (i + 1) * BATCH_UPSERT]

    vectors = [
        {'id': vid, 'values': backed_up[vid]['values'], 'metadata': backed_up[vid]['metadata']}
        for vid in batch_ids if vid in backed_up
    ]
    missing = [vid for vid in batch_ids if vid not in backed_up]
    if missing:
        missing_total += len(missing)

    if not vectors:
        continue

    result = upsert_with_backoff(index, vectors)
    total_upserted += result.upserted_count

elapsed = time.time() - t0
print(f'\n✅ {total_upserted} vectores subidos en {elapsed:.1f}s')
if missing_total:
    print(f'[aviso] {missing_total} nodos no tenian vector disponible (revisa la celda 8).')

## 11. Verificación del índice

In [ ]:
stats = index.describe_index_stats()
print('Estado del índice:')
print(f'  Vectores totales : {stats["total_vector_count"]}')
print(f'  Dimensión        : {stats["dimension"]}')

## 12. Test de consulta semántica

Prueba rápida que verifica que el índice responde correctamente.

In [ ]:
QUERY = 'Federal Reserve interest rate hike impact on stock market'

query_output = model.encode(
    [QUERY],
    max_length=MAX_LENGTH,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False,
)
query_vector = query_output['dense_vecs'][0].tolist()

results = index.query(vector=query_vector, top_k=5, include_metadata=True)
for match in results['matches']:
    md_ = match['metadata']
    print(f"{match['score']:.3f} | {md_.get('titulo', '')[:70]}")

## 13. Ejemplo de consulta con filtro de metadatos

Pinecone permite combinar búsqueda semántica con filtros exactos sobre metadata.

In [ ]:
QUERY_FILTERED = 'chip shortage supply chain disruption semiconductor'

query_output_f = model.encode(
    [QUERY_FILTERED],
    max_length=MAX_LENGTH,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False,
)
query_vector_f = query_output_f['dense_vecs'][0].tolist()

results_f = index.query(
    vector=query_vector_f,
    top_k=5,
    include_metadata=True,
    filter={'regimen_mercado': {'$eq': 'bearish'}},
)
for match in results_f['matches']:
    md_ = match['metadata']
    print(f"{match['score']:.3f} | {md_.get('titulo', '')[:70]}")

## 14. Restauración directa desde el backup (sin GPU)

Si el índice de Pinecone se borra o hay que recrearlo en otra cuenta/región,
**no hace falta volver a descargar el JSON de nodos ni tocar la GPU**: basta
con leer `BACKUP_FILE` y reinsertar los vectores ya calculados. Ejecuta esta
celda de forma independiente (solo necesita las celdas 2, 3 y 9 previas para
tener `Pinecone`, la configuración y el índice creado).

In [ ]:
def restore_from_backup(index, backup_file: str, batch_upsert: int = BATCH_UPSERT):
    restored = load_backup(backup_file)
    ids = list(restored.keys())
    n_batches = math.ceil(len(ids) / batch_upsert)
    total = 0

    print(f'Restaurando {len(ids)} vectores desde {backup_file}...')
    for i in tqdm(range(n_batches), desc='Restaurando', unit='batch'):
        batch_ids = ids[i * batch_upsert : (i + 1) * batch_upsert]
        vectors = [
            {'id': vid, 'values': restored[vid]['values'], 'metadata': restored[vid]['metadata']}
            for vid in batch_ids
        ]
        result = upsert_with_backoff(index, vectors)
        total += result.upserted_count

    print(f'✅ {total} vectores restaurados en "{INDEX_NAME}" sin usar la GPU.')


# Descomenta para ejecutar la restauracion:
# restore_from_backup(index, BACKUP_FILE)

## 15. Consolidación del backup a Parquet (opcional)

El backup en JSON Lines es óptimo mientras se genera (append incremental,
resistente a interrupciones), pero para almacenamiento a largo plazo o
para compartir el backup, `.parquet` es más compacto y rápido de cargar.
Esta celda convierte `BACKUP_FILE` a un único archivo `.parquet`.

In [ ]:
import pandas as pd

def consolidate_to_parquet(backup_file: str, parquet_file: str = 'noticias_embeddings_backup.parquet'):
    records = load_backup(backup_file)
    if not records:
        print('Backup vacio, nada que consolidar.')
        return None

    rows = []
    for rec in records.values():
        row = {'id': rec['id'], 'values': rec['values']}
        row.update(rec['metadata'])
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_parquet(parquet_file, index=False)
    print(f'✅ {len(df)} vectores consolidados en {parquet_file} '
          f'({Path(parquet_file).stat().st_size / 1024**2:.1f} MB)')
    return df


df_backup = consolidate_to_parquet(BACKUP_FILE)
if df_backup is not None:
    files.download('noticias_embeddings_backup.parquet')